# CPU launchable portfolio-optimization workflow

**CPU-only counterpart of an NVIDIA cuFOLIO notebook.** The matching unmodified GPU notebook is in `../upstream_notebooks/`. This version uses deterministic synthetic one-minute bars so it runs on GitHub Actions without NVIDIA infrastructure.

A compact end-to-end execution path: minute bars → daily returns → Mean–CVaR allocation → causal daily backtest. It deliberately replaces NVIDIA's GPU launchable environment with a normal GitHub-hosted CPU runner.

In [ ]:
from pathlib import Path
import sys

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src").exists():
        sys.path.insert(0, str(candidate / "src"))
        break

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from cufolio_cpu.returns import daily_returns_from_minute_bars
from cufolio_cpu.synthetic import synthetic_minute_bars

In [ ]:
bars = synthetic_minute_bars(sessions=45, seed=42)
daily_log, daily_simple = daily_returns_from_minute_bars(bars)
print(f"{len(bars):,} minute bars -> {daily_simple.shape[0]} daily sessions x {daily_simple.shape[1]} assets")
daily_simple.tail()

In [ ]:
from cufolio_cpu.optimize import mean_cvar_weights
from cufolio_cpu.backtest import walk_forward_rebalance

allocation = mean_cvar_weights(daily_simple, risk_aversion=5.0, max_weight=0.30)
performance, weights = walk_forward_rebalance(
    daily_simple, lookback_days=20, rebalance_every_days=5
)
print("Allocation status:", allocation.status)
print("Latest allocation:")
print(allocation.weights.sort_values(ascending=False).to_frame("weight"))
print("Final walk-forward equity:", f"{performance['equity'].iloc[-1]:.4f}")